In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx
od = sjx.optical_depth

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
xx = jnp.linspace(-9, 9, 2**13)

sigma_low = 1e-3
sigma_high = 1.0
gamma_low = 1e-3
gamma_high = 1.0
sigma = jnp.array([sigma_low, sigma_high, sigma_low, sigma_high])
gamma = jnp.array([gamma_low, gamma_low, gamma_high, gamma_high])

Voigt_dict = {}

Voigt_dict['scipy_wofz'] = od.voigt.log10_voigt(xx, sigma, gamma, mode="scipy_wofz")

voigt_convolution_batched = jit(
    vmap(lambda s, g: od.voigt.log10_voigt(xx, s, g, mode="convolution", oversample=4), in_axes=(0, 0))
)
Voigt_dict["JAX_convolution"] = voigt_convolution_batched(sigma, gamma)

voigt_exojax_batched = jit(
    vmap(lambda s, g: od.voigt.log10_voigt(xx, s, g, mode="ExoJAX"), in_axes=(0, 0))
)
Voigt_dict["exojax"] = voigt_exojax_batched(sigma, gamma)

In [ ]:
P = len(sigma)
colors = np.array(plt.cm.get_cmap("tab10").colors[:P])

fig, (ax, ax_res) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, gridspec_kw=dict(height_ratios=[2, 1]))

for ii in range(P):
    ax.plot(xx, 10**Voigt_dict['scipy_wofz'][ii], color=colors[ii], linestyle="-", linewidth=1.0)
    ax.plot(xx, 10**Voigt_dict['JAX_convolution'][ii], color=colors[ii], linestyle="--", linewidth=2.)
    ax.plot(xx, 10**Voigt_dict['exojax'][ii], color=colors[ii], linestyle=":", linewidth=3.5)

for ii in range(P):
    ax_res.plot(
        xx, 10**Voigt_dict['JAX_convolution'][ii] / 10**Voigt_dict['scipy_wofz'][ii] - 1,
        color=colors[ii], linestyle="--", linewidth=2.0
    )
    ax_res.plot(
        xx, 10**Voigt_dict['exojax'][ii] / 10**Voigt_dict['scipy_wofz'][ii] - 1,
        color=colors[ii], linestyle=":", linewidth=3.5
    )

method_handles = [
    Line2D([0], [0], color="k", lw=8,  ls="-",  label="Scipy wofz"),
    Line2D([0], [0], color="k", lw=4,  ls="--", label="JAX convolution"),
    Line2D([0], [0], color="k", lw=2,  ls=":",  label="exojax"),
]
leg1 = ax.legend(handles=method_handles, loc="upper right", fontsize=16)
ax.add_artist(leg1)

param_handles = []
for ii in range(P):
    lbl = rf"$\sigma={sigma[ii]:g},\ \gamma={gamma[ii]:g}$"
    param_handles.append(Line2D([0], [0], color=colors[ii], lw=6, label=lbl))

leg2 = ax.legend(handles=param_handles, loc="upper left", fontsize=16)
ax.add_artist(leg2)

ax.set_title("Voigt profiles")
ax.set_ylabel(r"$V(x;\ \sigma,\ \gamma)$")
ax.set_yscale("log")
ax.set_ylim(1e-6, 1e3)

ax_res.axhline(0.0, color="k", linewidth=1, alpha=0.5)
ax_res.set_xlabel("x")        # only bottom panel gets the x-label
ax_res.set_ylabel("residual")
ax_res.set_yscale("symlog", linthresh=1e-8)

major_ticks = [-1e-2, -1e-6, 0.0, 1e-6, 1e-2]
ax_res.set_yticks(major_ticks)

ax_res.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
ax_res.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

plt.subplots_adjust(hspace=0.05)
fig.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------
# Grid and parameters
# -----------------------------------------------
xx = jnp.linspace(-9.0, 9.0, 2**12)  # frequency (or velocity) offset grid

sigma0 = 0.2
gamma0 = 0.5

# -----------------------------------------------
# Helpers: scalar wrapper using the Voigt wrapper (ExoJAX mode)
# -----------------------------------------------
def voigt_exojax_scalar(x, sigma, gamma):
    """
    Scalar wrapper around the Voigt wrapper in ExoJAX mode
    so that we can use grad in the same style as the exojax example.
    """

    return 10**od.voigt.log10_voigt(jnp.atleast_1d(x), sigma, gamma, mode="ExoJAX")[0]

# Derivative wrt x (ν)
dVoigt_dx_exojax = jit(
    vmap(grad(voigt_exojax_scalar, argnums=0), (0, None, None), 0)
)

# Derivative wrt sigma (Gaussian width)
dVoigt_dsigma_exojax = jit(
    vmap(grad(voigt_exojax_scalar, argnums=1), (0, None, None), 0)
)

# Base Voigt profile from ExoJAX implementation via wrapper
Voigt_exojax_profile = jit(lambda x, s, g: 10**od.voigt.log10_voigt(x, s, g, mode="ExoJAX"))(
    xx, sigma0, gamma0
)

dV_dx_exojax = dVoigt_dx_exojax(xx, sigma0, gamma0)
dV_dsigma_exojax = dVoigt_dsigma_exojax(xx, sigma0, gamma0)

# -----------------------------------------------
# Autodiff through the JAX convolution Voigt (via wrapper)
# -----------------------------------------------
def voigt_conv_profile(sigma, gamma):
    """
    Voigt profile computed via JAX convolution on a fixed grid xx.
    This is fully JAX and supports autodiff.
    """
    return 10**od.voigt.log10_voigt(xx, sigma, gamma, mode="convolution", oversample=4)

# Profile for our reference parameters
Voigt_conv_profile = jit(voigt_conv_profile)(sigma0, gamma0)

# Derivative wrt sigma for the *entire profile* using jacfwd
dVoigtConv_dsigma = jit(jacfwd(lambda s: voigt_conv_profile(s, gamma0)))
dVConv_dsigma = dVoigtConv_dsigma(sigma0)  # shape (xx.size,)

# If you also want derivative wrt gamma:
dVoigtConv_dgamma = jit(jacfwd(lambda g: voigt_conv_profile(sigma0, g)))
dVConv_dgamma = dVoigtConv_dgamma(gamma0)

# -----------------------------------------------
# Optionally: use grad for a scalar functional of the profile
# (e.g. value at line center or integrated area)
# -----------------------------------------------
def voigt_conv_at_center(sigma, gamma):
    """Value of the convolved Voigt at x=0 (line center)."""
    prof = voigt_conv_profile(sigma, gamma)
    # xx is symmetric around 0; pick closest index to zero
    idx0 = xx.size // 2
    return prof[idx0]

d_center_dsigma = jit(grad(voigt_conv_at_center, argnums=0))(sigma0, gamma0)
d_center_dgamma = jit(grad(voigt_conv_at_center, argnums=1))(sigma0, gamma0)

print("d V_conv(x=0) / d sigma =", float(d_center_dsigma))
print("d V_conv(x=0) / d gamma =", float(d_center_dgamma))

# -----------------------------------------------
# Plotting example
# -----------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True)

ax = axes[0, 0]
ax.plot(xx, Voigt_exojax_profile, label="Voigt (ExoJAX mode)")
ax.plot(xx, Voigt_conv_profile, linestyle="--", label="Voigt (convolution mode)")
ax.set_ylabel("V(x)")
ax.set_title("Voigt profiles")
ax.legend()

ax = axes[1, 0]
ax.plot(xx, dV_dx_exojax, label=r"$\partial_x V_{\mathrm{ExoJAX}}$")
ax.set_xlabel("x")
ax.set_ylabel(r"dV/dx")
ax.set_title("Derivative wrt x (ExoJAX mode)")
ax.legend()

ax = axes[0, 1]
ax.plot(xx, dV_dsigma_exojax, label=r"$\partial_\sigma V_{\mathrm{ExoJAX}}$")
ax.plot(xx, dVConv_dsigma, linestyle="--", label=r"$\partial_\sigma V_{\mathrm{conv}}$")
ax.set_ylabel(r"dV/dσ")
ax.set_title("Derivative wrt σ")
ax.legend()

ax = axes[1, 1]
ax.plot(xx, dVConv_dgamma, label=r"$\partial_\gamma V_{\mathrm{conv}}$")
ax.set_xlabel("x")
ax.set_ylabel(r"dV/dγ")
ax.set_title("Derivative wrt γ (convolution)")
ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
from skewerjax.constants.lines import LINES

from dataclasses import dataclass

@dataclass(frozen=True)
class PhysicalConstants:
    c: float = 2.99792458e8                         # speed of light      — UNITS: [m / s]
    c_A_per_ps: float = 2997924.58                  # Speed of light      — UNITS: [Angstrom / ps]
    kB_amu: float = 8314.46262102654                # Boltzman constant   — UNITS: [amu ⋅ m^2 ⋅ s^-2 ⋅ K^-1]
    qe: float = 1.60217663e-19                      # electron charge     — UNITS: [C]
    me: float = 9.1093837e-31                       # electron mass       — UNITS: [kg]
    epsilon_0: float = 8.85418782e-12               # vacuum permittivity — UNITS: [C^2 ⋅ kg^−1 ⋅ m^−3 ⋅ s^2]

# single shared instance (immutable)
phys = PhysicalConstants()

In [ ]:
load_absorption_lines = ["HI-LyA", "HI-LyB", "SiIII-1206"]
absorption_lines = [LINES[lid] for lid in load_absorption_lines]
N_sp = len(absorption_lines) # number of absroption lines ''species'' (considering different absorption lines from different species at the same time)

# Build arrays of line parameters from Line objects
lambda0_A = jnp.array([line.lambda0_A for line in absorption_lines])  # (N_sp,) — UNITS: [Angs]
A21_THz   = jnp.array([line.A21_THz   for line in absorption_lines])  # (N_sp,) — UNITS: [THz]
mass_amu  = jnp.array([line.mass_amu  for line in absorption_lines])  # (N_sp,) — UNITS: [amu]
f_osc     = jnp.array([line.f_osc     for line in absorption_lines])  # (N_sp,) — UNITS: [adim]

# generate synthetic data for different skerwers
N_sk = 5                                                                              # number of skerwers
N_ppsk = 1                                                                            # number of points sampled per skerwer
tau_coll_ps = jnp.inf * jnp.ones((N_sp, N_sk, N_ppsk))                                # (N_sp, N_sk, N_ppsk) — UNITS: ns
b_turb_m_per_s = jnp.zeros((N_sp, N_sk, N_ppsk))                                      # (N_sp, N_sk, N_ppsk) — UNITS: m/s
T_K = jnp.ones((N_sp, N_sk, N_ppsk))*jnp.geomspace(1e2, 1e6, N_sk)[None, :, None]     # (N_sp, N_sk, N_ppsk) — UNITS: K
number_density_inv_nm3 = jnp.ones((N_sp, N_sk, N_ppsk))                               # (N_sp, N_sk, N_ppsk) — UNITS: nm^-3


gamma_THz = (A21_THz[:, None, None] + 2.0 / tau_coll_ps) / (4.0 * jnp.pi)  # (N_sp, 1, 1) — UNITS: [THz]

v_th_m_per_s = jnp.sqrt(2.0 * phys.kB_amu * T_K / mass_amu[:, None, None]) # (N_sp, N_sk, N_ppsk) — UNITS: [m/s]
v_eff_A_THz = (v_th_m_per_s + b_turb_m_per_s)*1e-2                         # (N_sp, N_sk, N_ppsk) — UNITS: [Å/ps]
sigma_THz = v_eff_A_THz / (2.0 * lambda0_A[:, None, None])                 # (N_sp, N_sk, N_ppsk) — UNITS: [THz]

In [ ]:
lambdas = jnp.linspace(970, 1280, 2**19)                                   # (N_lambda,) number of sampled wavelengths

nu_THz      = phys.c_A_per_ps / lambdas[None, :]                           # (1, N_lambda)    — UNITS: THz
nu0_THz     = phys.c_A_per_ps / lambda0_A[:, None]                         # (N_sp, 1)        — UNITS: THz
nu_grid_THz = nu_THz - nu0_THz                                             # (N_sp, N_lambda) — UNITS: THz. x_j(λ) for each line species j. Argument to Voigt for each line

In [ ]:
mode = "ExoJAX"

def log10_voigt_one_species(x_j, sigma_j, gamma_j): # sigma_j, gamma_j: (N_sk, N_ppsk)

    def log10_voigt_cell(s, g):
        return od.voigt.log10_voigt(x_j, s, g, mode=mode)                 # (N_lambda,)
    
    log10_voigt_over_pp = vmap(log10_voigt_cell, in_axes=(0, 0))          # vmap over ppsk inside skewer; (N_ppsk, N_lambda)
    log10_voigt_over_sk = vmap(log10_voigt_over_pp, in_axes=(0, 0))       # vmap over skewer; (N_sk, N_ppsk, N_lambda)

    return log10_voigt_over_sk(sigma_j, gamma_j)

log10_phi_ps = vmap(log10_voigt_one_species, in_axes=(0, 0, 0))(nu_grid_THz, sigma_THz, gamma_THz) # vmap over species j; (N_sp, N_sk, N_ppsk, N_lambda) — UNITS: [ps]

In [ ]:
cte_nm2_per_ps = (phys.qe**2 / (4 * phys.epsilon_0 * phys.me * phys.c)) * 1e6 # (N_sp, N_sk, N_ppsk, N_lambda) — UNITS: [nm^2 / ps]
sigma_nm2 = cte_nm2_per_ps * f_osc[:, None, None, None] * 10**log10_phi_ps    # (N_sp, N_sk, N_ppsk, N_lambda) — UNITS: [nm^2]
integrand_inv_nm = sigma_nm2 * number_density_inv_nm3[..., None]              # (N_sp, N_sk, N_ppsk, N_lambda) — UNITS: [nm^-1]

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(lambdas)           # (N_lambda,)
phi_np = np.asarray(10**log10_phi_ps)          # (N_sp, N_sk, N_ppsk, N_lambda)
T_vals = np.asarray(T_K[0, :, 0])          # (N_sk,)

colors = np.array(plt.cm.get_cmap("tab10").colors[:N_sk])
linestyles = ['-', '--', ':']


fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(absorption_lines):
    ax.axvline(line.lambda0_A, color="grey", linestyle="-.", linewidth=1.5, alpha=1.0)
    ls = linestyles[i_sp % len(linestyles)]
    for i_sk in range(N_sk):
        profile = phi_np[i_sp, i_sk, 0, :]   # full wavelength range
        ax.plot(lambdas_np, profile, color=colors[i_sk], linestyle=ls, linewidth=2.0)

# Axes formatting
ax.set_yscale("log")
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ps}\right]$")
ax.grid(alpha=0.2, which="both")
ax.set_title("Line profiles for different transitions and temperatures", fontsize=20)

# Legend for temperatures (colors)
temp_handles = []
for i_sk in range(N_sk):
    lbl = rf"$T = {T_vals[i_sk]:.2g}\,\mathrm{{K}}$"
    temp_handles.append(Line2D([0], [0], color=colors[i_sk], lw=4, linestyle="-", label=lbl))
leg_T = ax.legend(handles=temp_handles, loc="upper left", fontsize=16, ncols=1)
ax.add_artist(leg_T)

# Legend for lines (linestyles)
line_handles = []
for i_sp, line in enumerate(absorption_lines):
    ls = linestyles[i_sp % len(linestyles)]
    lbl = rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$"
    line_handles.append(Line2D([0], [0], color="k", lw=3, linestyle=ls, label=lbl))
leg_L = ax.legend(handles=line_handles, loc="lower right", fontsize=16, ncol=N_sp)

fig.tight_layout()
plt.show()



# Zoom half-width around each line center [Angstrom]
delta_lambda_A = 1.3   # adjust to taste
fig, axes = plt.subplots(nrows=1, ncols=N_sp, figsize=(14, 6), sharex=False, sharey=True)
if N_sp == 1:
    axes = [axes]  # in case N_sp=1, make it iterable

for i_sp, (ax, line) in enumerate(zip(axes, absorption_lines)):
    lambda0 = line.lambda0_A
    ax.axvline(lambda0, color="grey", linestyle="-.", linewidth=1.5)

    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]  # (N_lambda_window,)

    # Plot each temperature (each skewer)
    for i_sk in range(N_sk):
        profile = phi_np[i_sp, i_sk, 0, mask]  # (N_lambda_window,)
        ax.plot(lam_win, profile, color=colors[i_sk], linestyle=linestyles[i_sp], linewidth=2.0)

    # Panel title and scaling
    ax.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax.set_yscale("log")
    ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax.grid(alpha=0.2, which="both")
axes[0].set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ps}\right]$")

# Temperature legend (shared across panels)
temp_handles = []
for i_sk in range(N_sk):
    lbl = rf"$T = {T_vals[i_sk]:.2g}\,\mathrm{{K}}$"
    temp_handles.append(Line2D([0], [0], color=colors[i_sk], lw=5, label=lbl))
leg = axes[0].legend(handles=temp_handles, loc="lower left", fontsize=14, ncol=1)
axes[0].add_artist(leg)

fig.tight_layout()
plt.show()